# C1 — Cancellation Risk Classifier

Features (per spec): `delivery_zone` (encoded), `restaurant_name` (encoded), `order_value`, `discount_applied`, `delivery_time_mins` (fill missing with **zone median** before split).

Target: `1` if Cancelled, `0` if Delivered.

Model: RandomForest (`n_estimators=100`, `random_state=42`), 80/20 split.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

pd.set_option('display.max_columns', None)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/urbaneats_delivery_orders.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             150 non-null    object 
 1   order_date           150 non-null    object 
 2   restaurant_name      150 non-null    object 
 3   delivery_zone        150 non-null    object 
 4   order_value          150 non-null    float64
 5   delivery_time_mins   144 non-null    float64
 6   rider_rating         144 non-null    float64
 7   order_status         150 non-null    object 
 8   payment_method       150 non-null    object 
 9   discount_applied     150 non-null    float64
 10  customer_complaints  150 non-null    int64  
dtypes: float64(4), int64(1), object(6)
memory usage: 13.0+ KB


## Step 1 — Filter to Cancelled / Delivered and build target

In [7]:
train_df = df[df['order_status'].isin(["Cancelled", "Delivered"])].copy()
train_df['target'] = (train_df['order_status'] == "Cancelled").astype(int)
print(f"Rows used for training: {len(train_df)}")
print(train_df['target'].value_counts())

Rows used for training: 80
target
0    44
1    36
Name: count, dtype: int64


## Step 2 — Fill missing `delivery_time_mins` with **zone median** (before split)

In [8]:
zone_medians = train_df.groupby('delivery_zone')['delivery_time_mins'].transform('median')
print("Zone medians for delivery_time_mins:")
print(zone_medians)
train_df['delivery_time_mins'] = train_df['delivery_time_mins'].fillna(zone_medians)

# Also apply to the full 150-row df (needed for Step 13 scoring later)
full_zone_medians = df.groupby('delivery_zone')['delivery_time_mins'].transform('median')
df['delivery_time_mins'] = df['delivery_time_mins'].fillna(full_zone_medians)

print("Missing delivery_time_mins after fill (train):", train_df['delivery_time_mins'].isna().sum())
print("Missing delivery_time_mins after fill (full):", df['delivery_time_mins'].isna().sum())

Zone medians for delivery_time_mins:
1      53.5
3      53.5
5      63.5
6      53.5
7      57.0
       ... 
143    63.5
144    47.5
145    63.5
146    47.5
147    57.0
Name: delivery_time_mins, Length: 80, dtype: float64
Missing delivery_time_mins after fill (train): 0
Missing delivery_time_mins after fill (full): 0


## Step 3 — Encode `delivery_zone` and `restaurant_name`

Using the same integer mapping consistently for the training subset and the full 150-row dataset so that scoring in Step 13 lines up.

In [9]:
restaurant_map = {"Sushi Bay": 0, "Pizza Palace": 1, "Spice Garden": 2, "Burger Hub": 3, "Wrap & Roll": 4}
zone_map = {"East": 0, "North": 1, "West": 2, "Central": 3, "South": 4}

train_df['restaurant_enc'] = train_df['restaurant_name'].map(restaurant_map)
train_df['zone_enc'] = train_df['delivery_zone'].map(zone_map)

df['restaurant_enc'] = df['restaurant_name'].map(restaurant_map)
df['zone_enc'] = df['delivery_zone'].map(zone_map)

## Step 4 — Build feature matrix (only the 5 features the spec lists)

In [10]:
feature_cols = ['zone_enc', 'restaurant_enc', 'order_value', 'discount_applied', 'delivery_time_mins']
X = train_df[feature_cols]
y = train_df['target']
X.head()

,zone_enc,restaurant_enc,order_value,discount_applied,delivery_time_mins
1,0,1,807.0,12.0,53.5
3,0,4,675.0,3.0,53.5
5,2,1,814.0,24.0,63.5
6,0,1,423.0,2.0,53.0
7,1,3,134.0,1.0,80.0


## Step 5 — 80/20 split and train Random Forest

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## Step 6 — Evaluate: precision, recall, F1

In [12]:
y_pred = rf.predict(X_test)

print(f"Precision (Cancelled): {precision_score(y_test, y_pred):.3f}")
print(f"Recall    (Cancelled): {recall_score(y_test, y_pred):.3f}")
print(f"F1        (Cancelled): {f1_score(y_test, y_pred):.3f}")
print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=['Delivered', 'Cancelled']))

Precision (Cancelled): 0.500
Recall    (Cancelled): 0.250
F1        (Cancelled): 0.333

Full classification report:
              precision    recall  f1-score   support

   Delivered       0.50      0.75      0.60         8
   Cancelled       0.50      0.25      0.33         8

    accuracy                           0.50        16
   macro avg       0.50      0.50      0.47        16
weighted avg       0.50      0.50      0.47        16



## Step 7 — Feature importance: which restaurant / zone drives cancellation risk?

In [13]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)

restaurant_reverse = {v: k for k, v in restaurant_map.items()}
zone_reverse = {v: k for k, v in zone_map.items()}

cancel_by_restaurant = train_df.groupby('restaurant_name')['target'].mean().sort_values(ascending=False) * 100
cancel_by_zone = train_df.groupby('delivery_zone')['target'].mean().sort_values(ascending=False) * 100

print("\nCancellation rate by restaurant (%):")
print(cancel_by_restaurant.round(2))
print("\nCancellation rate by delivery zone (%):")
print(cancel_by_zone.round(2))

print(f"\nHighest-risk restaurant: {cancel_by_restaurant.idxmax()} ({cancel_by_restaurant.max():.2f}%)")
print(f"Highest-risk zone:       {cancel_by_zone.idxmax()} ({cancel_by_zone.max():.2f}%)")
print(f"\nrestaurant_enc importance: {importances['restaurant_enc']:.4f}")
print(f"zone_enc importance:       {importances['zone_enc']:.4f}")

Feature importances:
delivery_time_mins    0.306773
discount_applied      0.215118
order_value           0.207431
zone_enc              0.160958
restaurant_enc        0.109721
dtype: float64

Cancellation rate by restaurant (%):
restaurant_name
Wrap & Roll     60.00
Spice Garden    57.14
Burger Hub      50.00
Pizza Palace    40.00
Sushi Bay       22.22
Name: target, dtype: float64

Cancellation rate by delivery zone (%):
delivery_zone
Central    58.82
North      50.00
West       46.67
South      42.86
East       31.82
Name: target, dtype: float64

Highest-risk restaurant: Wrap & Roll (60.00%)
Highest-risk zone:       Central (58.82%)

restaurant_enc importance: 0.1097
zone_enc importance:       0.1610


### Business interpretation

Read the **printed output of the cell above** for the exact restaurant name, zone name, and percentages — markdown cells do not interpolate Python variables, so we summarise rather than embed.

The Random Forest feature importances show how strongly each feature influences the cancel/deliver prediction. Combined with the per-group cancellation rates, the restaurant and the delivery zone with the highest empirical cancellation rate are the operational hotspots.

**Business implication:** UrbanEats should prioritise rider allocation, kitchen prep monitoring, and customer communication for the highest-risk restaurant in the highest-risk zone, because that combination drives a disproportionate share of cancellations and directly erodes revenue per order.

## Step 8 — Score all 150 orders → `cancel_probability`, `cancel_risk`

In [14]:
X_all = df[feature_cols]
df['cancel_probability'] = rf.predict_proba(X_all)[:, 1]
df['cancel_risk'] = np.where(df['cancel_probability'] >= 0.55, 'high', 'low')

print(df['cancel_risk'].value_counts())
df[['order_id', 'restaurant_name', 'delivery_zone', 'order_status',
    'cancel_probability', 'cancel_risk']].head(10)

cancel_risk
low     101
high     49
Name: count, dtype: int64


,order_id,restaurant_name,delivery_zone,order_status,cancel_probability,cancel_risk
0,ORD00001,Pizza Palace,North,Delayed,0.25,low
1,ORD00002,Pizza Palace,East,Cancelled,0.15,low
2,ORD00003,Spice Garden,South,Delayed,0.55,high
3,ORD00004,Wrap & Roll,East,Delivered,0.31,low
4,ORD00005,Wrap & Roll,North,Refunded,0.39,low
5,ORD00006,Pizza Palace,West,Delivered,0.13,low
6,ORD00007,Pizza Palace,East,Cancelled,0.85,high
7,ORD00008,Burger Hub,North,Cancelled,0.67,high
8,ORD00009,Sushi Bay,East,Delivered,0.12,low
9,ORD00010,Sushi Bay,West,Delivered,0.02,low


In [15]:
df[df['cancel_risk'] == 'high'][['order_id', 'restaurant_name', 'delivery_zone',
                                  'order_status', 'cancel_probability']].sort_values(
    'cancel_probability', ascending=False)

,order_id,restaurant_name,delivery_zone,order_status,cancel_probability
90,ORD00091,Sushi Bay,Central,Cancelled,0.91
144,ORD00145,Spice Garden,South,Cancelled,0.88
140,ORD00141,Wrap & Roll,Central,Cancelled,0.87
84,ORD00085,Sushi Bay,Central,Cancelled,0.86
6,ORD00007,Pizza Palace,East,Cancelled,0.85
25,ORD00026,Wrap & Roll,Central,Cancelled,0.85
47,ORD00048,Burger Hub,Central,Cancelled,0.84
37,ORD00038,Burger Hub,West,Cancelled,0.84
66,ORD00067,Spice Garden,West,Cancelled,0.84
73,ORD00074,Wrap & Roll,North,Cancelled,0.83


# Step 9 - Export Scored CSV to drive

In [16]:
df.to_csv("/content/drive/MyDrive/Colab Notebooks/urbaneats_scored.csv", index=False)